# 1. Imports

In [75]:
import pandas as pd
from pathlib import Path
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

pd.set_option('display.max_columns', None)

# 2. Funções

## 2.1. Função para pegar os eventos de uma temporada nos arquivos parquet

In [76]:
def get_season_events_parquet_file_paths(events_competition_season_folder_path):
    
    season_events_parquet_file_paths = [
        str(Path(events_competition_season_folder_path) / season_event_parquet_file) 
        for season_event_parquet_file in os.listdir(events_competition_season_folder_path) 
        if season_event_parquet_file.endswith('.parquet')
        ]
    
    return season_events_parquet_file_paths

# 3. Preparação dos dados

## 3.1. Criação da Sessão Spark

In [77]:
# Criação da sessão Spark local
spark = SparkSession.builder.master("local[*]").appName("season_database").getOrCreate()

## 3.2. Criação do df para pegar os eventos de todas as partidas da temporada de 2022-2023 da Premier League

(dps pode ser interessante levar a parte do schema dos jogadores e da bola p etapa de extração)

In [78]:
events_competition_season_folder_path = str(Path().resolve().parent.parent / "data" / "events" / "1" / "2022-2023")

season_events_parquet_file_paths = get_season_events_parquet_file_paths(events_competition_season_folder_path)

# Criação do dataframe concatenando todos os arquivos parquet dos eventos das partidas entre as temporadas de todas as competições
df_events = spark.read.parquet(*season_events_parquet_file_paths)

df_events = df_events.withColumnsRenamed({
    "id": "eventId",
    "player.id": "eventPlayer.id",
    "player.name": "eventPlayer.name",
    "team.id": "eventTeam.id",
    "team.name": "eventTeam.name",
})

In [79]:
df_events.select('homePlayers', 'awayPlayers', 'balls').first()

Row(homePlayers='[{"speed": 0.452, "y": 25.001, "x": 15.746, "player": {"id": 292, "name": "Adam Smith"}, "visibility": "VISIBLE", "confidence": "LOW", "jerseyNum": null}, {"speed": 0.271, "y": -14.281, "x": 1.674, "player": {"id": 312, "name": "Dominic Solanke"}, "visibility": "VISIBLE", "confidence": "MEDIUM", "jerseyNum": null}, {"speed": 0.865, "y": -19.598, "x": 14.398, "player": {"id": 6994, "name": "Jordan Zemura"}, "visibility": "ESTIMATED", "confidence": "LOW", "jerseyNum": null}, {"speed": 0.163, "y": 5.119, "x": 8.692, "player": {"id": 7230, "name": "Ben Pearson"}, "visibility": "VISIBLE", "confidence": "MEDIUM", "jerseyNum": null}, {"speed": 0.792, "y": -3.978, "x": 18.401, "player": {"id": 289, "name": "Lloyd Kelly"}, "visibility": "ESTIMATED", "confidence": "LOW", "jerseyNum": null}, {"speed": 0.833, "y": 12.426, "x": 15.262, "player": {"id": 295, "name": "Jefferson Lerma"}, "visibility": "ESTIMATED", "confidence": "LOW", "jerseyNum": null}, {"speed": 0.775, "y": -8.477, 

In [80]:
events_competition_season_folder_path = str(Path().resolve().parent.parent / "data" / "events" / "1" / "2022-2023")

season_events_parquet_file_paths = get_season_events_parquet_file_paths(events_competition_season_folder_path)

# Criação do dataframe concatenando todos os arquivos parquet dos eventos das partidas entre as temporadas de todas as competições
df_events = spark.read.parquet(*season_events_parquet_file_paths)

df_events = df_events.withColumnsRenamed({
    "id": "eventId",
    "player.id": "eventPlayer.id",
    "player.name": "eventPlayer.name",
    "team.id": "eventTeam.id",
    "team.name": "eventTeam.name",
})

# Schema em Pyspark para poder parsear o json dos dados de tracking dos jogadores que está como string
players_schema = ArrayType(
    StructType([
        #StructField("speed", FloatType(), True),
        StructField("x", FloatType(), True),
        StructField("y", FloatType(), True),
        StructField("player", StructType([
            StructField("id", IntegerType(), True), 
            StructField("name", StringType(), True)]), 
            True),
        StructField("visibility", StringType(), True),
        StructField("confidence", StringType(), True),
        #StructField("jerseyNum", StringType(), True)      
    ])
)

# Schema em Pyspark para poder parsear o json dos dados de tracking da bola que está como string
balls_schema = ArrayType(
    StructType([
        StructField("x", FloatType(), True),
        StructField("y", FloatType(), True),
        StructField("z", FloatType(), True),
        StructField("visibility", StringType(), True)
    ])
)

details_schema = MapType(StringType(), StringType())

df_events = df_events.withColumns({
    # Cria coluna com json parseado para Lista de dicionários para dados de tracking do time mandante
    "homePlayers_parsed": F.from_json("homePlayers", players_schema),
    
    # Cria coluna com json parseado para Lista de dicionários para dados de tracking do time adversário
    "awayPlayers_parsed": F.from_json("awayPlayers", players_schema),

    # Cria coluna com json parseado para dicionário para dados de tracking da bola
    "balls_parsed": F.from_json("balls", balls_schema),

    "details_parsed": F.from_json("details", details_schema)

}).drop('homePlayers', 'awayPlayers', 'balls', 'details')

df_events = df_events.select(
    'competitionId',
    'season', # dps mudar pra seasonId se necessário
    'gameId',
    'eventId',
    'eventType',
    'eventTypeDescription',
    'period',
    #'periodDescription',
    #'startFormattedGameClock',
    'startGameClock',
    #'details_parsed',
    'homeTeam',
    F.col('`eventPlayer.id`').alias('eventPlayerId'),
    F.col('`eventPlayer.name`').alias('eventPlayerName'),
    F.col('`eventTeam.id`').alias('eventTeamId'), 
    F.col('`eventTeam.name`').alias('eventTeamName'), 
    'homePlayers_parsed', 
    'awayPlayers_parsed', 
    'balls_parsed'
)

In [81]:
df_events.select('homePlayers_parsed', 'awayPlayers_parsed', 'balls_parsed').first()

Row(homePlayers_parsed=[Row(x=15.746000289916992, y=25.000999450683594, player=Row(id=292, name='Adam Smith'), visibility='VISIBLE', confidence='LOW'), Row(x=1.6740000247955322, y=-14.281000137329102, player=Row(id=312, name='Dominic Solanke'), visibility='VISIBLE', confidence='MEDIUM'), Row(x=14.39799976348877, y=-19.597999572753906, player=Row(id=6994, name='Jordan Zemura'), visibility='ESTIMATED', confidence='LOW'), Row(x=8.692000389099121, y=5.11899995803833, player=Row(id=7230, name='Ben Pearson'), visibility='VISIBLE', confidence='MEDIUM'), Row(x=18.400999069213867, y=-3.9779999256134033, player=Row(id=289, name='Lloyd Kelly'), visibility='ESTIMATED', confidence='LOW'), Row(x=15.26200008392334, y=12.425999641418457, player=Row(id=295, name='Jefferson Lerma'), visibility='ESTIMATED', confidence='LOW'), Row(x=-0.15299999713897705, y=-8.47700023651123, player=Row(id=7966, name='Kieffer Moore'), visibility='VISIBLE', confidence='LOW'), Row(x=39.97700119018555, y=-0.12099999934434891,

In [82]:
print('Quantidade de linhas:', df_events.count())

Quantidade de linhas: 945154


## 3.2. Obter jogos da temporada e ajustar identificação do mandante/adversário

(dps pode ser interessante levar isso p etapa de extração)

In [83]:
games_path = str(Path().resolve().parent.parent / "data" / "games.csv")

df_games = spark.read.csv(games_path, header=True)

df_games_raw = df_games.withColumnRenamed("id","gameId").filter(F.col('season') == '2022-2023')

# se venueType == TEAM_HOME, (homeTeam.id == team.id e homeTeam.name == team.name) e (opponentTeam.id == opponentTeam.id e opponentTeam.name == opponentTeam.name)
# se venueType == OPPONENT_HOME, (homeTeam.id == opponentTeam.id e homeTeam.name == opponentTeam.name) e (opponentTeam.id == team.id e opponentTeam.name == team.name)
df_games = (
    df_games_raw.withColumns({
    "homeTeamId": F.when(F.col('venueType') == 'TEAM_HOME', F.col('`team.id`')).otherwise(F.col('`opponentTeam.id`')),
    "homeTeamName": F.when(F.col('venueType') == 'TEAM_HOME', F.col('`team.name`')).otherwise(F.col('`opponentTeam.name`')),

    "opponentTeamId": F.when(F.col('venueType') == 'TEAM_HOME', F.col('`opponentTeam.id`')).otherwise(F.col('`opponentTeam.id`')),
    "opponentTeamName": F.when(F.col('venueType') == 'TEAM_HOME', F.col('`opponentTeam.name`')).otherwise(F.col('`opponentTeam.name`')),
    }).select(
        'gameId', 
        'date',
        'season',
        F.col('`competition.id`').alias('competitionId'),
        F.col('`competition.name`').alias('competitionName'),
        'homeTeamId',
        'homeTeamName',
        'opponentTeamId',
        'opponentTeamName',
        #F.col('teamExtraTimeStartSide').alias('homeTeamExtraTimeStartSide'), 
        F.col('teamStartSide').alias('homeTeamStartSide'),
        F.col('`stadium.name`').alias('stadiumName'), 
        F.col('`stadium.length`').cast("float").alias('stadiumLength'), 
        F.col('`stadium.width`').cast("float").alias('stadiumWidth')
    )
)

df_games.show(5)

+------+----------+---------+-------------+---------------+----------+--------------------+--------------+--------------------+-----------------+-------------+-------------+------------+
|gameId|      date|   season|competitionId|competitionName|homeTeamId|        homeTeamName|opponentTeamId|    opponentTeamName|homeTeamStartSide|  stadiumName|stadiumLength|stadiumWidth|
+------+----------+---------+-------------+---------------+----------+--------------------+--------------+--------------------+-----------------+-------------+-------------+------------+
|  4447|2022-08-13|2022-2023|            1| Premier League|         3|         Aston Villa|             8|             Everton|             Left|   Villa Park|        105.0|        68.0|
|  4760|2023-04-25|2022-2023|            1| Premier League|        20|Wolverhampton Wan...|            20|Wolverhampton Wan...|             Left|     Molineux|        105.0|        68.0|
|  4451|2022-08-15|2022-2023|            1| Premier League|      

## 4. Junção dos dados dos Jogos + Eventos em uma tabela

In [84]:
df_games_events = df_events.join(df_games.drop('season', 'competitionId', 'competitionName'), on = "gameId", how='left')

df_games_events = (
    df_games_events
    .withColumn(
        'homeTeamAttackDirection',
            F.when(
                ((F.col('period') == 1) & (F.col('homeTeamStartSide') == 'Right')) | 
                ((F.col('period') == 2) & (F.col('homeTeamStartSide') == 'Left')), 
                'Left'
            )
            .when(
                ((F.col('period') == 1) & (F.col('homeTeamStartSide') == 'Left')) | 
                ((F.col('period') == 2) & (F.col('homeTeamStartSide') == 'Right')), 
                'Right'
            )
        )
    .withColumn(
            'awayTeamAttackDirection',
            F.when(F.col('homeTeamAttackDirection') == 'Right', 'Left')
            .when(F.col('homeTeamAttackDirection') == 'Left', 'Right')
        )
    .drop('homeTeamStartSide')
)

df_games_events.show(5)

+------+-------------+---------+--------------------+------------+--------------------+------+--------------+--------+-------------+---------------+-----------+-------------+--------------------+--------------------+--------------------+----------+----------+---------------+--------------+----------------+----------------+-------------+------------+-----------------------+-----------------------+
|gameId|competitionId|   season|             eventId|   eventType|eventTypeDescription|period|startGameClock|homeTeam|eventPlayerId|eventPlayerName|eventTeamId|eventTeamName|  homePlayers_parsed|  awayPlayers_parsed|        balls_parsed|      date|homeTeamId|   homeTeamName|opponentTeamId|opponentTeamName|     stadiumName|stadiumLength|stadiumWidth|homeTeamAttackDirection|awayTeamAttackDirection|
+------+-------------+---------+--------------------+------------+--------------------+------+--------------+--------+-------------+---------------+-----------+-------------+--------------------+-----

In [85]:
df_games_events.groupby('period').count().show()

+------+------+
|period| count|
+------+------+
|     1|479855|
|     2|465299|
+------+------+



In [86]:
# variável que indica o time com a posse
df_games_events.groupBy('homeTeam').count().show()

+--------+------+
|homeTeam| count|
+--------+------+
|    NULL|  6918|
|    true|472671|
|   false|465565|
+--------+------+



In [87]:
df_games_events = (
    df_games_events
    .filter(F.col('period').isin([1,2])) # filtro para garantir apenas eventos das partidas no 1º e 2º tempo
    .dropna(subset='homeTeam') # drop nos eventos onde nenhum dos dois times tem a posse    
    )

## Normalização do ataque sempre pra direita

In [88]:
# time com a posse está atacando e time sem está defendendo

df_games_events_tracking = (
    df_games_events
    # time atacando = se o time da casa tiver a posse, pega tracking home, se não pega tracking away
    .withColumn(
        "attackingPlayers",
        F.when(F.col("homeTeam"), F.col("homePlayers_parsed"))
        .otherwise(F.col("awayPlayers_parsed"))
    )
    # time defendendo = se o time da casa tiver a posse, pega tracking away, se não pega tracking home
    .withColumn(
        "defendingPlayers",
        F.when(F.col("homeTeam"), F.col("awayPlayers_parsed"))
        .otherwise(F.col("homePlayers_parsed"))
    )
    .withColumn(
        "attackingDirection",
        F.when(F.col("homeTeam"), F.col("homeTeamAttackDirection"))
        .otherwise(F.col("awayTeamAttackDirection"))
    )
    # flag para normalização (para tratar ataque sempre pra direita)
    .withColumn(
        "need_side_revert",
        F.col("attackingDirection") == "Left"
    )
    .drop(
        'homePlayers_parsed',  
        'awayPlayers_parsed',
        'homeTeamAttackDirection',
        'awayTeamAttackDirection'
    )
)

df_games_events_tracking.show()

+------+-------------+---------+--------------------+------------+--------------------+------+--------------+--------+-------------+----------------+-----------+---------------+--------------------+----------+----------+---------------+--------------+----------------+----------------+-------------+------------+--------------------+--------------------+------------------+----------------+
|gameId|competitionId|   season|             eventId|   eventType|eventTypeDescription|period|startGameClock|homeTeam|eventPlayerId| eventPlayerName|eventTeamId|  eventTeamName|        balls_parsed|      date|homeTeamId|   homeTeamName|opponentTeamId|opponentTeamName|     stadiumName|stadiumLength|stadiumWidth|    attackingPlayers|    defendingPlayers|attackingDirection|need_side_revert|
+------+-------------+---------+--------------------+------------+--------------------+------+--------------+--------+-------------+----------------+-----------+---------------+--------------------+----------+---------

In [89]:
players_tracking_norm = (
        lambda p: F.struct(
            # reversão do eixo horizontal qnd necessário
            F.when(F.col("need_side_revert"), -p["x"])
            .otherwise(p["x"])
            .alias("x"),
            # variáveis restantes mantém igual
            p["y"].alias("y"),            
            p["player"].alias("player"),
            p["visibility"].alias("visibility"),
            p["confidence"].alias("confidence")
        )
)

balls_norm = (
    F.transform(
        "balls_parsed",
        lambda b: F.struct(
            # reversão do eixo horizontal qnd necessário
            F.when(F.col("need_side_revert"), -b["x"])
            .otherwise(b["x"])
            .alias("x"),
            # variáveis restantes mantém igual            
            b["y"].alias("y"),
            b["z"].alias("z"),            
            b["visibility"].alias("visibility")
        )
    )
)

# df para normalizar os atacantes, defensores e bola sempre atacando do lado direito
df_games_events_tracking_norm = (
    df_games_events_tracking
    # normalizar atacantes
    .withColumns({
        "attackingPlayersNorm": F.transform("attackingPlayers", players_tracking_norm),
        # normalizar defensores
        "defendingPlayersNorm": F.transform("defendingPlayers", players_tracking_norm),
        # normalizar a bola
        "ballsNorm": balls_norm
    }).drop(
        'attackingPlayers', 
        'defendingPlayers',
        'balls_parsed',
        'attackingDirection',
        'need_side_revert'
        )
)

In [92]:
df_games_events_tracking_norm.show()

+------+-------------+---------+--------------------+------------+--------------------+------+--------------+--------+-------------+----------------+-----------+---------------+----------+----------+---------------+--------------+----------------+----------------+-------------+------------+--------------------+--------------------+--------------------+
|gameId|competitionId|   season|             eventId|   eventType|eventTypeDescription|period|startGameClock|homeTeam|eventPlayerId| eventPlayerName|eventTeamId|  eventTeamName|      date|homeTeamId|   homeTeamName|opponentTeamId|opponentTeamName|     stadiumName|stadiumLength|stadiumWidth|attackingPlayersNorm|defendingPlayersNorm|           ballsNorm|
+------+-------------+---------+--------------------+------------+--------------------+------+--------------+--------+-------------+----------------+-----------+---------------+----------+----------+---------------+--------------+----------------+----------------+-------------+------------

In [ ]:
# TO-DO:
## métrica 1 de ameaça de avanço territorial
## métrica 2 de ameaça
## métrica 3 de ameaça
## validação das 3 métricas
## normalização das 3 métricas 
## criar ameaça = média das 3 métricas
## criar delta de ameaça
## filtrar apenas os eventos defensivos


In [ ]:
# event_id-season-gameId-match_id-possession_team-attacking_team-defending_team-ball_x-ball_y-attacking_players-defending_players-threat-threat_delta

## Domínios

### Tipos de eventos:

- FIRSTKICKOFF: Inicio do primeiro tempo
- SECONDKICKOFF: Inicio do segundo tempo
- TC: Touch
- RE: Rebound
- BC: Ball Carry
- CL: Clearance
- CR: Cross
- CH: Challenge 
- OTB: A possession with a player on the ball
- PA: Pass
- FO: Foul
- FOUL: Additional foul
- SH: Shot

### Domínio: Desempenho Técnico Defensivo

### Eventos que queremos (Defensivos):

- Ofensivos como CR, PA e SH queremos que o resultado dele seja uma interferência da defesa adversária
- Defensivos como CL, CH, FO queremos que o tipo seja ação defensiva

- CL: Clearance
    - Qualquer CLEARANCE_OUTCOME_TYPE (A,B,D,E,O,P,S,U)
    - obs: talvez não E e U pq são FairPlay
    - obs2: P - Player e S - Stoppage não sei oq sejam, mas vou deixar

- CR: Cross
    - CROSS_OUTCOME_TYPE:
    - B - Blocked
    - D - Defensive Interception

- CH: Challenge. 
    - CHALLENGE_TYPE:
    - ‘5’ - 50/50. This is a duel type where two players compete for a loose ball.
    - A - Aerial duel. As the name suggests a duel type similar to 50-50, but with the ball coming from above.
    - B - Tackle from behind. As the name suggests a tackle attempt where the carrier puts their body in between the ball and the challenger as the tackle is attempted.
    - D - Dribble. The player tries to take on a defender in an attempt to get past them.
    - G - Goalkeeper smothers ball. A duel between the goalkeeper and a line player where the ball is loose and the goalkeeper tries to capture the ball.
    - H - Shielding. Similar to tackle from behind, but on a shielding challenge the carrier actively shields a defender who does not attempt a tackle
    - K - Hand tackle by goalkeeper. Despite the name, it is a duel type similar to goalkeeper smothers, but the keeper tries to parry the ball rather than retain it.
    - L - Slide tackle. Tackle type where the challenger slides to attempt to win the ball. Note that a player could be sliding on a dribble or 50-50, to be classed as a slide tackle it needs to be first and foremost a tackle.
    - S - Shoulder to shoulder. Tackle type where the challenger tries to win the ball with physical contact initiated with the body.
    - T - Standing tackle. Tackle attempt, usually from the front or side, that does not fit the other tackle types 
    - OBS1: **Único que não entraria como AD aqui seria o 'D'.**
    - OBS2: **Não estamos considerando outcome dos eventos.**

- PA: Pass
    - PASS_OUTCOME_TYPE:
    - B - Blocked
    - D - Defensive Interception

- FO: Foul
    - qualquer FOUL_TYPE = A, I, M
    - não vi evento de penalti, então teria q pegar a região dentro da area e evento de falta marcado ali (FOUL_TYPE == I)

- FOUL: Additional foul
    - são faltas adicionais no mesmo lance divida em mais de um evento, mas nos dados fica tudo NULL, então n vou add. FO já tem o evento principal de falta

- SH: Shot
    - SHOT_OUTCOME_TYPE:
    - B - Block on target. (Ball was going on target, but got blocked)
    - C - Block off target. (Ball was going off target, and got blocked)
    - F - Save off target. (Ball was going off target when it got saved)
    - L - Goalline clearance. (Ball is past the goalkeeper and a defender stops it from going into the net)
    - S - Save on target. (Ball was going on target and got saved).

### Domínio: Ameaça

Vamos criar as variáveis de ameaça para os dois times, respeitando os sentidos de ataques deles. Isso apenas para os eventos defensivos.

In [ ]:
# eventos com posse pega só do time com a posse
# eventos sem posse, pega dos dois e faz a média

#### Métrica 1: Distância percorrida no campo (medida pela menor distância entre os escanteios)

#### Métrica 2: Quantidade total de jogadores dos dois times (entre a bola e o gol)

#### Métrica 3: Diferencial da quantidade de defensores e atacantes